<a href="https://colab.research.google.com/github/aWolander/google-colab/blob/main/Federated%20Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from __future__ import annotations
import copy
import torch
import time
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets
from torchvision.transforms import ToTensor, Normalize, Compose, Resize, CenterCrop
import matplotlib.pyplot as plt
import random
import numpy as np
import math
import gc
from tqdm.notebook import trange, tqdm
try:
    device = torch.accelerator.current_accelerator().type if \
    torch.accelerator.is_available() else "cpu" # i get an error when i run this locally
except:
    device="cpu"

transform = Compose([
    #Resize(224), # slows down process significantly
    #CenterCrop(224),
    ToTensor(),
    Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
]) # https://github.com/facebookresearch/dino/blob/7c446df5b9f45747937fb0d72314eb9f7b66930a/eval_image_retrieval.py#L106

CIFAR100 = datasets.CIFAR100(
  root="data",
  download=True,
  transform=transform
)


100%|██████████| 169M/169M [00:02<00:00, 64.0MB/s]


In [ ]:
def main() -> None:
    """
    Loads CIFAR-100 dataset and splits it into training, validation, and test sets.
    Then performs an IID client split and tests the data distribution.
    """
    K = 5  # number of clients
    N = 10
    CIFAR100 = datasets.CIFAR100(
        root="data",
        download=True,
        transform=ToTensor()
    )

    train_dataset, validate_dataset, test_dataset = torch.utils.data.random_split(
        CIFAR100, [0.7, 0.15, 0.15]
    )

    client_datasets = split_data_non_iid(train_dataset, K, N)
    test_split(client_datasets)


def create_label_indexing(dataset: Dataset) -> dict[int, list[int]]:
    """
    Creates a dictionary mapping each class label to the list of indices where it appears.

    Args:
        dataset: Dataset object (e.g. CIFAR100 or Subset).

    Returns:
        A dictionary mapping label -> list of indices.
    """
    label_index = {i: [] for i in range(100)}
    for idx, (_, label) in enumerate(dataset):
        label_index[label].append(idx)
    return label_index


def split_data_non_iid(dataset: Dataset, K: int, N_c: int = 100) -> list[Subset]:
    """
    Splits the dataset into K non-IID subsets based on label distribution.

    Args:
        dataset: PyTorch dataset.
        K: Number of clients.
        N_c: Number of labels per client.

    Returns:
        List of K Subsets representing client datasets.
    """
    index_split = [[] for _ in range(K)]
    label_index = create_label_indexing(dataset)
    unused_labels = list(range(100))
    available_labels = {}

    for client in range(K):
        client_labels = random.sample(unused_labels, N_c)
        available_labels[client] = client_labels

    labels_exhausted = [False for _ in range(K)]

    while not all(labels_exhausted):
        for client in range(K):
            if not available_labels[client]:
                labels_exhausted[client] = True
                continue
            for label in available_labels[client][:]:
                if not label_index[label]:
                    available_labels[client].remove(label)
                    continue
                index_split[client].append(label_index[label].pop())

    for indices in index_split:
        random.shuffle(indices)

    return [Subset(dataset, indices) for indices in index_split]


def split(l: list[int], n: int) -> list[list[int]]:
    """
    Splits a list into n parts in a round-robin fashion.

    Args:
        l: List to split.
        n: Number of splits.

    Returns:
        A list of n sublists.
    """
    return [l[i::n] for i in range(n)]


def split_data_iid(dataset: Dataset, K: int) -> list[Subset]:
    """
    Splits the dataset IID among K clients.

    Args:
        dataset: PyTorch dataset.
        K: Number of clients.

    Returns:
        List of K Subsets representing IID client datasets.
    """
    index_split = [[] for _ in range(K)]
    label_index = create_label_indexing(dataset)

    for label in range(100):
        split_indices = split(label_index[label], K)
        for client in range(K):
            index_split[client] += split_indices[client]

    for indices in index_split:
        random.shuffle(indices)

    return [Subset(dataset, indices) for indices in index_split]


def test_split(client_datasets: list[Subset]) -> None:
    """
    Plots the label distribution across client datasets.

    Args:
        client_datasets: List of Subsets, each representing a client.

    Returns:
        None
    """
    bottom = np.zeros(100)

    for client_id, dataset in enumerate(client_datasets):
        occurrences = np.zeros(100)
        for _, label in dataset:
            occurrences[label] += 1

        non_zero = occurrences[occurrences > 0]
        print(
            f"Client {client_id}: "
            f"Classes = {np.count_nonzero(occurrences)}, "
            f"Mean = {non_zero.mean():.2f}, Std = {non_zero.std():.2f}"
        )

        plt.bar(range(100), occurrences, bottom=bottom, label=f"Client {client_id}")
        bottom += occurrences

    plt.xlabel("Class label")
    plt.ylabel("Number of samples")
    plt.title("Client Data Distribution")
    plt.show()

#if "__main__" == __name__:
#    main()



In [ ]:
'''This is done seperately so that saving and loading the models does not grant
a different dataset. A bit annoying to have to change K and N_c here but what can you do'''
K=100
N_c = 10

train_dataset, validate_dataset, test_dataset = torch.utils.data.random_split(CIFAR100, [0.7,0.15,0.15])

split_dataset_iid = split_data_iid(train_dataset, K)
split_dataset_non_iid = split_data_non_iid(train_dataset, K, N_c)
split_train_dataloader_iid = [DataLoader(subset, batch_size=100) for subset in split_dataset_iid]
split_train_dataloader_non_iid = [DataLoader(subset, batch_size=100) for subset in split_dataset_non_iid]


train_dataloader = DataLoader(train_dataset, batch_size=100)
test_dataloader = DataLoader(test_dataset, batch_size=100)
validate_dataloader = DataLoader(validate_dataset, batch_size=100)

In [ ]:
#import logging

#logging.basicConfig(level=logging.INFO)
#logger = logging.getLogger(__name__)

class DinoFullModel(nn.Module):
    """
    A wrapper around the DINO ViT-S/16 model.
    Adds a custom classification head and includes training and evaluation utilities.
    """

    def __init__(self, learning_rate: float = 1e-3, momentum=0.9, filepath: str|None = None) -> None:
        """
        Adds a custom classification head and includes training and evaluation utilities.
        Initializes the model with a DINO backbone and custom classification head.

        Args:
            learning_rate (float): Learning rate for optimizer.
            epochs (int): Number of training epochs.
        """
        super().__init__()

        self.learning_rate = learning_rate
        self.momentum = momentum



        # Load pretrained DINO model
        self.backbone = torch.hub.load('facebookresearch/dino:main', 'dino_vits16',_verbose=False).to(device)
        # Replace head with a new classifier for CIFAR
        self.head = nn.Linear(self.backbone.embed_dim, 100)
        self.loss_fn = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.SGD(self.parameters(), lr=self.learning_rate, momentum = self.momentum)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=30)
        self.handles = []


        self.scores = {}
        self.grad_masks = []
        self.masked_named_parameters = []
        self.original_parameters = {}

        for name, param in self.backbone.named_parameters():
            self.masked_named_parameters.append((torch.ones_like(param), name, param))


        # self.backbone.head = nn.Sequential( # apparantly not necessary
        #     nn.Linear(self.backbone.embed_dim, 256),
        #     nn.ReLU(inplace=True),
        #     nn.Dropout(0.5),
        #     nn.Linear(256, 128),
        #     nn.ReLU(inplace=True),
        #     nn.Dropout(0.5),
        #     nn.Linear(128, 100)
        # ).to(device)

        # Loss and optimizer

        # For gradient masking


        # Performance tracking
        self.training_loss_history = []
        self.training_accuracy_history = []
        self.validate_loss_history = []
        self.validate_accuracy_history = []
        if filepath:
            self.load_model(filepath)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))

    def train_model(self, dataloader: DataLoader, epochs, validate_dataloader: DataLoader|None = None, filepath: str|False=False) -> None:
        """
        Trains the model with optional validation and checkpointing.
        """
        self.train()
        running_loss, total_correct, total_samples = 0.0, 0, 0
        dataset_size = len(dataloader.dataset)
        for epoch in tqdm(range(1, epochs+1), desc="Epochs", leave=None):
            running_loss, total_correct, total_samples = 0.0, 0, 0

            if validate_dataloader:
                val_loss, val_acc = self.test_model(validate_dataloader)
                self.validate_loss_history.append(val_loss)
                self.validate_accuracy_history.append(val_acc)
                print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")

            #for X, y in tqdm(dataloader, desc="Batches", leave=None):
            for X, y in dataloader:

                X, y = X.to(device), y.to(device)

                outputs = self(X)
                self.loss = self.loss_fn(outputs, y)

                self.loss.backward()
                self.optimizer.step()
                #self.zero_grad()
                self.optimizer.zero_grad()


                running_loss += self.loss.item() * X.size(0)
                total_correct += (outputs.argmax(dim=1) == y).sum().item()
                total_samples += X.size(0)

            epoch_loss = running_loss / total_samples
            epoch_acc = total_correct / total_samples
            self.training_loss_history.append(epoch_loss)
            self.training_accuracy_history.append(epoch_acc)

            print(f"Train Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")


            self.scheduler.step()
            if filepath and epoch%10==0:
                self.save_model(filepath)

    def TaLoS(self, \
                dataloader: DataLoader, \
                rounds:int, \
                sparsity: float, \
                filepath: str|False=False, \
                q:float = 0) -> None:
        '''
        https://arxiv.org/pdf/2504.02620
        '''
        self.freeze_head()
        self.unfreeze_backbone()
        self.eval()

        if sparsity < 1.0:
            for round in tqdm(range(1, rounds + 1), desc="Finetuning rounds"):
                sparseness = sparsity ** (round / rounds)
                #for (mask, name, param), param_other in zip(self.masked_named_parameters, self.backbone.parameters()):
                #    print(f"1 {id(param) == id(param_other)}")
                self.score_parameters(dataloader)
                if q >= 1:
                    self.test_mask_quantization(sparseness, q)
                #for (mask, name, param), param_other in zip(self.masked_named_parameters, self.backbone.parameters()):
                #    print(f"2 {id(param) == id(param_other)}")
                if filepath:
                    self.save_model(filepath)
                print("Scoring complete, creating mask.")

                self.create_mask(sparseness)
                self.drop_parameters()
        #self.backbone = torch.hub.load('facebookresearch/dino:main', 'dino_vits16').to(device)
        self.recover_parameters()
        #for (mask, name, param), param_other in zip(self.masked_named_parameters, self.backbone.parameters()):
        #    print(f"3 {id(param) == id(param_other)}")
        self.apply_gradient_mask()


    def score_parameters(self, dataloader: DataLoader) -> None:
        self.scores = {}
        for mask, name, parameter in self.masked_named_parameters:
            self.scores[name] = torch.zeros_like(parameter)

        total_samples = 0
        dataset_size = len(dataloader.dataset)

        time_spent = 0.0

        for X, y in tqdm(dataloader,desc="Finetuning batches",leave=None):
            X, y = X.to(device), y.to(device)
            total_samples += X.size(0)
            logits = self(X)
            outdx = torch.distributions.Categorical(logits=logits).sample().unsqueeze(1).detach()
            samples = logits.gather(1, outdx)

            for datapoint_index in tqdm(range(X.size(0)),desc="Progress in batch", leave=None):
                self.zero_grad()
                torch.autograd.backward(samples[datapoint_index], retain_graph=True)
                for mask, name, parameter in self.masked_named_parameters:
                    if name in self.scores and parameter.grad is not None:
                        self.scores[name] += torch.clone(parameter.grad.data.pow(2).detach())


    @torch.no_grad()
    def create_mask(self, sparsity):
        scores = torch.cat([torch.flatten(v) for v in self.scores.values()])
        masks = torch.cat([torch.flatten(m).cpu() for m, n, p in self.masked_named_parameters if n in self.scores])
        t, _ = torch.kthvalue(scores[masks == 1.0], min(int(sparsity * scores[masks == 1.0].numel()), scores[masks == 1.0].numel() - 1))
        print('[+] Threshold:', t)
        tot, not_masked = 0, 0
        for mask, name, param in self.masked_named_parameters:
            if name in self.scores:
                score = self.scores[name]
                score[mask != 1.0] = torch.inf
                final_score = torch.ones_like(score)
                final_score.mul_(torch.where(score > t, 0.0, 1.0))
                mask.copy_(torch.where(score > t, 0.1, 1.0))
                tot += final_score.numel()
                not_masked += final_score.sum().item()
                setattr(param, 'score', final_score.clone().detach().cpu())
        print('[+] Remaining weights:', not_masked / tot, not_masked, tot)
        gc.collect()
        torch.cuda.empty_cache()

    def apply_gradient_mask(self):
        for mask, name, param in self.masked_named_parameters:
            mask = mask.to(param.device)
            handle = param.register_hook(self.gradient_mask_hook(mask))
            self.handles.append(handle)

    @torch.no_grad()
    def drop_parameters(self):
        for mask, name, param in self.masked_named_parameters:
            if name not in self.original_parameters:
                self.original_parameters[name] = param.data.clone().cpu()
            mask = mask.to(param.device)
            param.data.mul_(mask)

    def recover_parameters(self):
        for name, param in self.backbone.named_parameters():
            if name in self.original_parameters:
                original_data = self.original_parameters[name].to(param.device)
                param.data.copy_(original_data)
            else:
                print(name)


    def remove_previous_mask(self):
        for handle in self.handles:
            handle.remove()
        self.handles = []

    def gradient_mask_hook(self, m):
        return lambda grad: (grad * m)


    def test_model(self, dataloader: DataLoader) -> tuple[float, float]:
        """
        Evaluates the model on a test/validation dataloader.
        """
        self.eval()
        total_loss, total_correct, total_samples = 0.0, 0, 0

        with torch.no_grad():
            for X, y in dataloader:
                X, y = X.to(device), y.to(device)
                outputs = self(X)
                loss = self.loss_fn(outputs, y)

                total_loss += loss.item() * X.size(0)
                total_correct += (outputs.argmax(dim=1) == y).sum().item()
                total_samples += X.size(0)

        return total_loss / total_samples, total_correct / total_samples

    def get_quantized_scores(self, q):
        # Flatten all scores into one vector
        flattened_scores = torch.cat([s.flatten() for s in self.scores.values()])
        abs_scores = flattened_scores.abs()
        #x_max = abs_scores.max()
        #x_min = abs_scores.min()

        # Vectorized quantization
        quantized_scores_dict = {}
        for param_name, score_matrix in self.scores.items():
            abs_vals = score_matrix.abs()
            x_max_param = abs_vals.max()
            x_min_param = abs_vals.min()
            signs = score_matrix.sign()

            normalized = (abs_vals - x_min_param) / (x_max_param - x_min_param + 1e-6)
            normalized_clipped = torch.clamp(normalized, 0, 0.999999)

            # Compute l and probabilities
            l = (normalized_clipped * q).floor()
            prob = normalized_clipped * q - l

            # Stochastic rounding
            rand_vals = torch.rand_like(prob)
            round_up = (rand_vals < prob).float()
            phi_vals = (l + round_up) / q

            # Final quantized values
            quantized = signs * (x_min_param + (x_max_param - x_min_param) * phi_vals)

            quantized_scores_dict[param_name] = quantized

        return quantized_scores_dict

    @torch.no_grad()
    def test_mask_quantization(self, sparsity:float, q:float) -> None:
        quantized_scores_dict = self.get_quantized_scores(q)

        scores = torch.cat([torch.flatten(v) for v in self.scores.values()])
        quantized_scores = torch.cat([torch.flatten(v) for v in quantized_scores_dict.values()])

        masks = torch.cat([torch.flatten(m).cpu() for m, n,  p in self.masked_named_parameters if n in self.scores])


        t, _ = torch.kthvalue(scores[masks == 1.0], min(int(sparsity * scores[masks == 1.0].numel()), scores[masks == 1.0].numel() - 1))
        t_q, _ = torch.kthvalue(quantized_scores[masks == 1.0], min(int(sparsity * quantized_scores[masks == 1.0].numel()), quantized_scores[masks == 1.0].numel() - 1))


        total_different_parameters = 0
        # Iterate through parameters that were considered for pruning (mask was not 0)
        for mask, name, param in self.masked_named_parameters:
            if name in self.scores:
                score = self.scores[name].clone()
                qscore = quantized_scores_dict[name]

                score[mask != 1.0] = torch.inf
                qscore[mask != 1.0] = torch.inf

                final_score = torch.ones_like(score)
                final_score.mul_(torch.where(score > t, 0.0, 1.0))

                final_qscore = torch.ones_like(qscore)
                final_qscore.mul_(torch.where(qscore > t_q, 0.0, 1.0))

                different_parameters = torch.count_nonzero(abs(final_score - final_qscore)).item()
                total_different_parameters += different_parameters


        print(f"different_parameters: {total_different_parameters}")
        changed_parameters = min(int(sparsity * scores[masks == 1.0].numel()), scores[masks == 1.0].numel() - 1)
        # Calculate percentage of different decisions relative to the number of parameters considered for pruning
        if changed_parameters > 0:
             print(f"Percent wrong: {total_different_parameters / changed_parameters}")
        else:
            print("Percent wrong: N/A (No parameters considered for pruning)")

        print(f"threshold values: {t} and {t_q}")

    def plot_performance(self) -> None:
        """
        Plots training and validation loss and accuracy curves.
        """
        fig, axs = plt.subplots(1, 2, figsize=(12, 5))
        if self.training_loss_history:
            axs[0].plot(self.training_loss_history, label='Train Loss')
        if self.validate_loss_history:
            axs[0].plot(self.validate_loss_history, label='Val Loss')
        axs[0].set_title('Loss')
        axs[0].legend()
        axs[0].grid(True)
        if self.training_accuracy_history:
            axs[1].plot(self.training_accuracy_history, label='Train Accuracy')
        if self.validate_accuracy_history:
            axs[1].plot(self.validate_accuracy_history, label='Val Accuracy')
        axs[1].set_title('Accuracy')
        axs[1].legend()
        axs[1].grid(True)

        hyperparam_text = (
        f"Learning rate: {self.learning_rate}\n"
        f"Momentum: {self.optimizer.param_groups[0].get('momentum', 'N/A')}\n"
        f"Scheduler: CosineAnnealingLR\n"
        #f"Epochs: {len(self.training_loss_history)}"
        )
        # Put it in the top left of the first subplot
        axs[0].text(
            0.02, 0.98, hyperparam_text,
            transform=axs[0].transAxes,
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='lightgray', alpha=0.5)
        )

        plt.tight_layout()
        plt.show()


    def save_model(self, filepath: str) -> None:
        """
        Saves model weights and optimizer state.

        Args:
            filepath (str): Path to save the checkpoint.
        """
        torch.save({
            'model_state_dict': self.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'training_loss_history': self.training_loss_history,
            'training_accuracy_history': self.training_accuracy_history,
            'validate_loss_history': self.validate_loss_history,
            'validate_accuracy_history': self.validate_accuracy_history,
            'scores': self.scores,
            'masked_named_parameters': self.masked_named_parameters, # Save masks and parameter ids
            'original_parameters': self.original_parameters # Save original parameters

        }, filepath)
        print(f"Model saved to {filepath}")

    def recover_model(self, filepath: str) -> None:
        """
        Loads model weights and optimizer state from a checkpoint.

        Args:
            filepath (str): Path to the saved checkpoint.
        """
        checkpoint = torch.load(filepath, map_location=device)
        self.load_state_dict(checkpoint['model_state_dict'])

        #self.handles = checkpoint['handles']
        self.scores = checkpoint['scores']
        self.masked_named_parameters = checkpoint['masked_named_parameters']
        self.original_parameters = checkpoint['original_parameters'] # Load original parameters

        self.training_loss_history = checkpoint['training_loss_history']
        self.training_accuracy_history = checkpoint['training_accuracy_history']
        self.validate_loss_history = checkpoint['validate_loss_history']
        self.validate_accuracy_history = checkpoint['validate_accuracy_history']
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        # Move optimizer state to the correct device
        for state in self.optimizer.state.values():
            for k, v in state.items():
                if isinstance(v, torch.Tensor):
                    state[k] = v.to(device)

        print(f"Model recovered from {filepath}")

    def load_model(self, filepath: str) -> None:
        """
        Loads model weights from a checkpoint.f

        Args:
            filepath (str): Path to the saved checkpoint.
        """
        checkpoint = torch.load(filepath, map_location=device)
        self.load_state_dict(checkpoint['model_state_dict'])
        self.scores = checkpoint['scores']
        self.masked_named_parameters = checkpoint['masked_named_parameters']

        self.unfreeze_head()
        self.unfreeze_backbone()
        self.refresh_optimizer()
        self.masked_named_parameters = []
        for (name, param), (mask, _, old_param) in zip(self.backbone.named_parameters(), checkpoint['masked_named_parameters']):
            self.masked_named_parameters.append((mask, name, param))


    def refresh_optimizer(self):
        self.optimizer = torch.optim.SGD(self.parameters(), lr=self.learning_rate, momentum=self.momentum)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=30) # Also re-initialize scheduler


    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad_(False)

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad_(True)

    def freeze_head(self):
        for param in self.head.parameters():
            param.requires_grad_(False)

    def unfreeze_head(self):
        for param in self.head.parameters():
            param.requires_grad_(True)

    def apply_mask(self):
        self.apply_gradient_mask()
        self.drop_parameters()

In [ ]:
class FL_client(DinoFullModel):
    '''
    Basically just the single model, but with addition and scaling functionality.
    To facilitate FedAvg more easily.
    '''
    def __init__(self) -> None:
        super().__init__()


    def __add__(self, other: FL_client) -> FL_client:
        assert isinstance(other, FL_client)
        temp_client = FL_client().to(device) # there has to be a better way
        temp_client.load_state_dict(self.state_dict())
        with torch.no_grad():
            for (name, param), (_, other_param) in zip(temp_client.named_parameters(), other.named_parameters()):
                param.data.copy_(param.data + other_param.data.to(device))
        return temp_client

    def __mul__(self, multiplier: float|int) -> FL_client:
        assert isinstance(multiplier, float|int)
        temp_client = FL_client().to(device)
        temp_client.load_state_dict(self.state_dict())
        with torch.no_grad():
                for name, param in temp_client.named_parameters():
                    param.data.copy_(param * multiplier)
        return temp_client

    def __rmul__(self, multiplier: float|int) -> FL_client:
        return self.__mul__(multiplier)

    def __sub__(self, other: FL_client) -> FL_client:
        assert isinstance(other, FL_client)
        return self + (-other)

    def __truediv__(self, divisor: float|int) -> FL_client:
        return self.__mul__(1/divisor)

    def get_model(self, other: FL_client) -> None:
        assert isinstance(other, FL_client)
        with torch.no_grad():
            for (name, param), (_, other_param) in zip(self.named_parameters(), other.named_parameters()):
                param.data.copy_(other_param.data)


class FL_server():
    def __init__(self, K: int, filepath:str|None = None) -> None:
        self.model = DinoFullModel().to(device)

        self.K = K

        self.loss_fn = nn.CrossEntropyLoss()
        self.clients = []
        for i in range(self.K):
            temp_client = FL_client().to(device)
            self.clients.append(temp_client) # pointers? should be fine

        if filepath:
            self.load_model(filepath)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    def FedAvg(self, dataloaders_list: list[DataLoader], \
               FedAvg_rounds: int, \
               client_epochs: int, \
               C: float, \
               validate_dataloader: DataLoader|None = None, \
               filepath: str|None = None) -> None:

        for FedAvg_round in tqdm(range(1, FedAvg_rounds+1), desc="FedAvg round", leave=None):
            m = max(int(C*self.K), 1)
            rand_set_clients = random.sample(range(self.K), m)
            client_model_sum = None
            m_t = 0
            for client_id in tqdm(rand_set_clients, desc="Client", leave=None):
                temp_client = self.clients[client_id]

                temp_loader = dataloaders_list[client_id]

                temp_client.train_model(temp_loader, client_epochs)

                n_k = len(temp_loader.dataset)
                m_t += n_k
                # Easier than instantiating an empty model
                if client_model_sum is None:
                    client_model_sum = n_k*temp_client
                else:
                    client_model_sum = client_model_sum + n_k*temp_client

            client_model_sum = (1/m_t) * client_model_sum
            for (name, param), (_, other_param) in zip(self.model.named_parameters(), client_model_sum.named_parameters()):
                with torch.no_grad():
                    param.copy_(other_param)

            for client in self.clients:
                client.get_model(client_model_sum)

            if validate_dataloader:
                val_loss, val_acc = self.test_model(validate_dataloader)
                self.model.validate_loss_history.append(val_loss)
                self.model.validate_accuracy_history.append(val_acc)
                print(f"Validation Loss For Server: {val_loss:.4f}, Accuracy: {val_acc:.4f}")
            torch.cuda.empty_cache()



            if filepath and FedAvg_round%10==0:
                self.plot_performance()
                self.save_model(filepath)

    def distributed_TaLoS(self,
                          dataloaders_list: list[DataLoader], \
                          rounds:int, \
                          sparsity:float, \
                          filepath:str|None = None) -> None:
        '''
        https://arxiv.org/pdf/2504.02620
        '''
        for client_id, client in tqdm(enumerate(self.clients), desc="Finetuning clients"):
            client.TaLoS(dataloaders_list[client_id], rounds, sparsity)
            #client.save_model(f"filepath{client_id}")
            gc.collect()
            torch.cuda.empty_cache()

    def shared_mask_TaLoS(self, dataloaders_list: list[DataLoader], \
                sparsity:float, \
                rounds:int,\
                C: float, \
                q=0,\
                filepath: str|False=False \
                ) -> None:
        '''
        https://arxiv.org/pdf/2504.02620
        '''
        for round in tqdm(range(1,rounds+1), desc="Finetuning rounds"):
            sparseness = sparsity ** (round/rounds)
            m_t = 0.0
            client_scores_sum = {}
            m = max(int(C*self.K), 1)
            rand_set_clients = random.sample(range(self.K), m)
            for client_id in tqdm(rand_set_clients, desc="Finetuning for clients"):
                temp_client = self.clients[client_id]
                temp_client.score_parameters(dataloaders_list[client_id])


                loader = dataloaders_list[client_id]
                temp_parameters = temp_client.backbone.parameters()
                temp_scores = temp_client.scores
                if q >= 1:
                    temp_scores = temp_client.get_quantized_scores(q)
                    temp_client.test_mask_quantization(sparseness, q) # we quantize twice for this. computationally wasteful but whatever
                n_k = len(loader.dataset)
                m_t += n_k
                for name, score in temp_scores.items():
                    client_scores_sum[name] = n_k*score + client_scores_sum.get(name, 0)

            for name, score in client_scores_sum.items():
                client_scores_sum[name] *= 1/m_t

            if filepath:
                self.model.scores = client_scores_sum
                self.save_model(filepath)
            for client in self.clients:
                client.scores = client_scores_sum # pointers shared here. Should not be a problem
                client.create_mask(sparseness)
                client.drop_parameters()

        for client in self.clients:
            client.recover_parameters()
            client.apply_gradient_mask

    def test_model(self, dataloader: DataLoader) -> tuple[float, float]:
        #print(f"Validation Loss: {avg_loss:.4f}, Correct: {total_correct} out of {len(dataloader)}, Accuracy: {accuracy:.2%}")
        return self.model.test_model(dataloader)

    def plot_performance(self) -> None:
        self.model.plot_performance()

    def save_model(self, filepath: str) -> None:
        """
        Saves model weights and optimizer state.

        Args:
            filepath (str): Path to save the checkpoint.
        """
        self.model.save_model(f"{filepath}")
        print(f"Server saved to {filepath}")

    def load_model(self, filepath: str) -> None:
        """
        Loads model weights from a checkpoint.

        Args:
            filepath (str): Path to the saved checkpoint.
        """
        self.model.load_model(f"{filepath}")
        print(f"Server Model loaded from {filepath}")

    def recover_model(self, filepath: str) -> None:
        """
        Loads model weights and optimizer state from a checkpoint.

        Args:
            filepath (str): Path to the saved checkpoint.
        """
        self.model.recover_model(f"{filepath}")
        print(f"Server Model recovered from {filepath}")

    def load_single_model(self,filepath: str) -> None:
        for client_id in range(self.K):
            self.clients[client_id].load_model(f"{filepath}")
        print(f"Clients and server Model loaded from {filepath}")




## Main

In [ ]:
'''
TODO:
Batch normalization, probably. I think I read somewhere that the facebook model needs this. someone also needs to look up how to make good use of the model
    - Added normalization transform to images from facebook github // Gustav
Version control and checkpointing.
    - added version control, checkpoitning left //Gustav
    - Added checkpointing. Not tested yet //Gustav
    - Tested and fully implemented // Gustav
Testing and hyperparameter tuning. also model head architecture
    - Added very simple model head architecture
gradient mask TaLoS thing. Also expanding this to FL. This is a big one.
    - possibly implemented for single model case. needs testing
Plot accuracy and loss method in DinoFullModel
    - added // Gustav
Type hinting
    - added // Gustav

Questions:
    - Is the sharding supposed to work like this?
        - is the iid supposed to have a uniform distribution of labels
        - explain non-iid
    - Is there a better way to do addition and multiplication in FedAvg?
    - Facebook model head. Is it correct? Should the backend be static?
    - are the results reasonable?
    - Batch normalization?
    - The aldready implemented normalization?

Personal contribution:
    - Quantization
    - Normalising parameters. they get very large. dont know why
    - could be done layer by layer so that a roughly even amount of parameters
        get pruned
    - sharing gradient mask
'''

gc.collect()
torch.cuda.empty_cache()
def create_classifier():
    model = DinoFullModel(learning_rate=1e-3, momentum = 0.9).to(device)
    model.freeze_backbone()
    model.train_model(train_dataloader, 30, validate_dataloader=validate_dataloader, filepath="classifier")
    model.plot_performance()
    print("Done!")

def single_test():
    model = DinoFullModel(learning_rate=1e-3, momentum = 0.9).to(device)

    model.train_model(train_dataloader, 30, validate_dataloader=validate_dataloader, filepath="lr3m9e30cosine")
    model.plot_performance()

def finetune_test_single():
    path = "lr3m9e30cosine_sp1r3_1"
    R = 0.9
    model = DinoFullModel(learning_rate=1e-4, momentum = 0.9, filepath="classifier").to(device)
    model.TaLoS(train_dataloader, 3, R, q=50, filepath="test")
    model.train_model(train_dataloader, 30, validate_dataloader, filepath="test")
    model.plot_performance()


def FL_test_iid(J):
    gc.collect()
    torch.cuda.empty_cache()
    FL = FL_server(K)
    #FL.load_single_model("baseline")
    FL.FedAvg(split_train_dataloader_iid, \
              FedAvg_rounds = 50, client_epochs = J, C= 0.1, \
              validate_dataloader=validate_dataloader, \
              filepath="test_FLmodel")
    print(FL.test_model(test_dataloader))
    FL.plot_performance()

def FL_test_non_iid(J):
    gc.collect()
    torch.cuda.empty_cache()
    FL = FL_server(K)
    #FL.recover_model(filepath="test_FLmodel")
    #FL.plot_performance()
    #FL.load_single_model("baseline")
    FL.FedAvg(split_train_dataloader_non_iid, \
              FedAvg_rounds = 50, client_epochs = J, C= 0.1, \
              validate_dataloader=validate_dataloader, \
              filepath="test_FLmodel")
    print(FL.test_model(test_dataloader))
    FL.plot_performance()
    print(J)

def FL_test_finetune():
    FL = FL_server(K)
    FL.load_single_model("classifier")
    FL.distributed_TaLoS(split_train_dataloader_non_iid, \
                         sparsity=0.9, rounds = 1, \
                         filepath="test_FLmodel_finetune")
    FL.FedAvg(split_train_dataloader_non_iid, \
              FedAvg_rounds = 50, client_epochs = 4, C= 0.1, \
              validate_dataloader=validate_dataloader, \
              filepath="test_FLmodel_finetune")
    FL.plot_performance()


def FL_test_finetune_share(q=-1):
    FL = FL_server(K)
    FL.load_single_model("classifier")
    FL.shared_mask_TaLoS(split_train_dataloader_iid, \
                         sparsity=0.1, rounds = 4, C=0.1,\
                         q=q,\
                         filepath="test_FLmodel_shared_finetune")

    FL.FedAvg(split_train_dataloader_iid, \
              FedAvg_rounds = 50, client_epochs = 4, C= 0.1, \
              validate_dataloader=validate_dataloader, \
              filepath="test_FLmodel_shared_finetune")
    FL.plot_performance()

def test_quantizer():
    model = DinoFullModel().to(device)
    arr = torch.randn(3,3)
    model.scores = [arr]
    print(model.scores)
    print(model.get_quantized_scores(2))

#create_classifier()
#finetune_test_single()
#FL_test_finetune()
FL_test_finetune_share()


This is just for testing

In [ ]:
# Test for apply_gradient_mask
def test_apply_gradient_mask(dummy_model = None):

    if dummy_model is None:
        dummy_model = DinoFullModel().to(device)
        with torch.no_grad():
            for mask, name, param in dummy_model.masked_named_parameters:
                random_mask = torch.randn_like(param.data)

                mask.copy_(torch.where(random_mask > 0, 0.1, 1.0))


    # Apply the gradient mask
    test_datapoint = torch.randn(1,3,32,32).to(device)

    temp_output = dummy_model(test_datapoint)

    dummy_model.loss = dummy_model.loss_fn(temp_output, torch.tensor([0]).to(device))
    dummy_model.loss.backward()
    gradients_before_mask = []
    for param in dummy_model.parameters():
        gradients_before_mask.append(param.grad.clone())
    dummy_model.zero_grad()

    dummy_model.apply_gradient_mask()

    temp_output = dummy_model(test_datapoint)

    dummy_model.loss = dummy_model.loss_fn(temp_output, torch.tensor([0]).to(device))
    dummy_model.loss.backward()

    # Check if the gradient is masked correctly
    for (mask,_, param), before_grad in zip(dummy_model.masked_named_parameters, gradients_before_mask):
        expected_masked_grad = before_grad * mask.to(device)
        assert torch.allclose(expected_masked_grad, param.grad), "Gradient not masked correctly"

    # Clean up hooks
    dummy_model.remove_previous_mask()

test_apply_gradient_mask()
print("apply_gradient_mask test passed.")

In [ ]:
# Test for drop_parameters
def test_drop_parameters(dummy_model = None):

    if dummy_model is None:
        dummy_model = DinoFullModel(filepath="classifier").to(device)
        with torch.no_grad():
            for mask, name, param in dummy_model.masked_named_parameters:
                random_mask = torch.randn_like(param.data)

                mask.copy_(torch.where(random_mask > 0, 0.1, 1.0))
    #for mask, param in dummy_model.masked_parameters:
    #    print(param.name)
    original_parameters = {}
    for name, param in dummy_model.backbone.named_parameters():
        #print(name)
        original_parameters[name] = param.data.clone().to(device)


    # Drop parameters
    dummy_model.drop_parameters()

    # Check if parameters are dropped correctly
    for (mask, name, param), param_other in zip(dummy_model.masked_named_parameters, dummy_model.backbone.parameters()):
        expected_param = original_parameters[name].to(device) * mask.to(device)
        assert torch.allclose(param, param_other), f"Parameters not dropped correctly for {param.name}"
        #assert torch.allclose(param.data, expected_param), f"Parameters not dropped correctly for {param.name}"

    dummy_model.recover_parameters()

    # Check if parameters are restored correctly
    for name, param in dummy_model.backbone.named_parameters():
        assert torch.allclose(param.data, original_parameters[name]), f"Parameters not restored correctly for {name}"

    for mask, name, param in dummy_model.masked_named_parameters:
        print(param.requires_grad)

test_drop_parameters()
print("drop_parameters test passed.")